In [6]:
import pandas as pd
import numpy as np
import h5py
import datetime
from datetime import datetime
from datetime import timedelta 
import time

# Data processing

In [9]:
cleaned_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED_OUTLIER.h5'
with h5py.File(cleaned_path, 'r') as hdf:
    keys = list(hdf.keys())
    print(keys)

['table_data']


In [ ]:
chunk_size = 13_000_000  # Adjust the chunk size as needed
df_data = pd.DataFrame()

# Read the filtered data in chunks  from the HDF5 file

with pd.HDFStore(cleaned_path, mode='r') as store:
    nrows = store.get_storer('table_data').nrows
    if nrows is not None:
        total_chunks = nrows // chunk_size + 1
    else:
        total_chunks = 0
    print(f"Rows in table_data: {nrows}\n")
    start_time, step_time = time.time(), time.time()
    for i, chunk in enumerate(store.select('table_data', chunksize=chunk_size)):
        if df_data.empty:
            df_data = chunk
        else:
            df_data = pd.concat([df_data, chunk], ignore_index=True)  # Concatenate chunk to df_data_filtered (inplace=True)
        del chunk  # Delete the chunk variable to free up memory
        print(f"Processed chunk {i + 1} of {total_chunks} | Time taken: {(time.time() - step_time):.1f} seconds")
        step_time = time.time()
print(f"Total time taken:  {(time.time() - start_time):.1f} seconds")
del store, total_chunks
df_data

Rows in table_data: 123607762

Processed chunk 1 of 10 | Time taken: 12.2 seconds


In [ ]:
df_data['year'] = pd.to_datetime(df_data['datetime']).dt.year
df_p_availability = df_data.groupby(['gauge_code', 'year']).agg({'rain_mm': 'count'}).reset_index()
df_p_availability['days_in_year'] = df_p_availability.apply(lambda x: 365 if (x['year'] % 4 != 0 or (x['year'] % 100 == 0 and x['year'] % 400 != 0)) else 366, axis=1)
df_p_availability['p_availability'] = df_p_availability['rain_mm'] / df_p_availability['days_in_year'] * 100
df_p_availability['p_availability'] = df_p_availability['p_availability'].fillna(0).replace([np.inf, -np.inf], 0)
df_p_availability = df_p_availability[['gauge_code', 'year', 'p_availability']]
df_p_availability

,gauge_code,year,p_availability
0,00047000,1961,100.00000
1,00047000,1962,100.00000
2,00047000,1963,100.00000
3,00047000,1964,100.00000
4,00047002,1977,6.30137
...,...,...,...
346024,S713,2021,100.00000
346025,S714,2021,100.00000
346026,S715,2021,100.00000
346027,S716,2021,100.00000


In [45]:
df_p_availability.to_hdf(cleaned_path
                         , key = 'table_p_availability'
                         , encoding = 'utf-8'
                         , mode='r+'
                         , format='table'
                         , complevel=9
                         , append=False)


df_p_availability = pd.read_hdf(cleaned_path, key='table_p_availability')
df_p_availability

,gauge_code,year,p_availability
0,00047000,1961,100.00000
1,00047000,1962,100.00000
2,00047000,1963,100.00000
3,00047000,1964,100.00000
4,00047002,1977,6.30137
...,...,...,...
346024,S713,2021,100.00000
346025,S714,2021,100.00000
346026,S715,2021,100.00000
346027,S716,2021,100.00000


In [46]:
with h5py.File(cleaned_path, 'r') as hdf:
    keys = list(hdf.keys())
    print(keys)

['table_data', 'table_data_filtered', 'table_info', 'table_p_availability', 'table_preclassif', 'table_q1_gaps', 'table_q2_week', 'table_q3_outliers', 'table_qc_info']
